In [1]:
import pandas as pd
df = pd.read_csv('consumer_data.csv')
df['ItemEncoded'] = df['Item'].astype('category').cat.codes
df['PaymentEncoded'] = df['Mode of Payment'].astype('category').cat.codes
df['Price'] = (df['Price'] - df['Price'].mean()) / df['Price'].std()
df['Quantity'] = (df['Quantity'] - df['Quantity'].mean()) / df['Quantity'].std()
df['Price'] = df['Price'].astype(float)
df['Quantity'] = df['Quantity'].astype(float)
print(df.head(10))
print(df.info())
df_processed = df.drop(['Date', 'Item', 'Mode of Payment'], axis=1)

C:\Users\srnav\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\srnav\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


        Date          Item     Price  Quantity Mode of Payment  ItemEncoded  \
0   4/1/2022          eggs -0.625197 -1.405138      Debit Card           32   
1   7/1/2022          eggs -0.625197 -0.701733            Cash           32   
2   7/1/2022         bread -0.288216  0.001673            Cash            9   
3   7/1/2022          rice  0.975463 -1.405138            Cash           98   
4   8/1/2022        yogurt -0.709442 -1.405138            Cash          122   
5   8/1/2022  granola bars -0.161848 -0.701733            Cash           63   
6   9/1/2022     mushrooms -0.498829 -0.701733      Debit Card           80   
7  10/1/2022    sauerkraut -0.077603 -0.701733     Credit Card          102   
8  11/1/2022          milk -0.077603 -0.701733     Credit Card           78   
9  11/1/2022         bread -0.288216 -0.701733     Credit Card            9   

   PaymentEncoded  
0               2  
1               0  
2               0  
3               0  
4               0  
5         

In [2]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
latent_dim = 32
num_features = 4  # Exclude 'Date' column
generator = keras.Sequential([
    layers.Dense(128, activation='relu', input_dim=latent_dim),
    layers.Dense(256, activation='relu'),
    layers.Dense(num_features, activation='tanh')
])

discriminator = keras.Sequential([
    layers.Dense(256, activation='relu', input_dim=num_features),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Compile discriminator
discriminator.compile(optimizer='adam', loss='binary_crossentropy')

# Create GAN
discriminator.trainable = False
gan_input = keras.Input(shape=(latent_dim,))
gan_output = discriminator(generator(gan_input))
gan = keras.Model(gan_input, gan_output)
gan.compile(optimizer='adam', loss='binary_crossentropy')


batch_size = 32
epochs = 1000
for epoch in range(epochs):
    noise = np.random.normal(0, 1, size=(batch_size, latent_dim))
    generated_data = generator.predict(noise)

    real_data = df.drop(['Date', 'Item', 'Mode of Payment'], axis=1).sample(batch_size)
    real_labels = np.ones((batch_size, 1))
    fake_labels = np.zeros((batch_size, 1))

    # Train discriminator
    d_loss_real = discriminator.train_on_batch(real_data, real_labels)
    d_loss_fake = discriminator.train_on_batch(generated_data, fake_labels)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    # Train generator (via GAN)
    noise = np.random.normal(0, 1, size=(batch_size, latent_dim))
    g_loss = gan.train_on_batch(noise, real_labels)

    # Print progress
    if epoch % 100 == 0:
        print(f"Epoch: {epoch}, D Loss: {d_loss}, G Loss: {g_loss}")


1/1 [==============================] - 0s 213ms/step
Epoch: 0, D Loss: 3.773520976305008, G Loss: 0.7001121044158936
1/1 [==============================] - 0s 43ms/step
Epoch: 100, D Loss: 0.06155654415488243, G Loss: 3.3568477630615234
1/1 [==============================] - 0s 26ms/step


1/1 [==============================] - 0s 40ms/step
Epoch: 200, D Loss: 0.1233559399843216, G Loss: 3.8087806701660156
1/1 [==============================] - 0s 40ms/step
Epoch: 300, D Loss: 0.009389069289682084, G Loss: 4.028247833251953
1/1 [==============================] - 0s 40ms/step


1/1 [==============================] - 0s 34ms/step
Epoch: 400, D Loss: 0.0012631913138196893, G Loss: 6.367151260375977
1/1 [==============================] - 0s 28ms/step


1/1 [==============================] - 0s 42ms/step
Epoch: 500, D Loss: 0.0008920964284591659, G Loss: 6.338208198547363
1/1 [==============================] - 0s 41ms/step
Epoch: 600, D Loss: 0.006143583275843412, G Loss: 4.593456268310547
1/1 [==============================] - 0s 29ms/step


1/1 [==============================] - 0s 36ms/step
Epoch: 700, D Loss: 0.02253449536692642, G Loss: 5.801055908203125
1/1 [==============================] - 0s 45ms/step


1/1 [==============================] - 0s 41ms/step
Epoch: 800, D Loss: 0.00019891364618196405, G Loss: 7.788332939147949
1/1 [==============================] - 0s 42ms/step
Epoch: 900, D Loss: 0.014604286174289882, G Loss: 3.8629679679870605
1/1 [==============================] - 0s 28ms/step


1/1 [==============================] - 0s 41ms/step


In [4]:
df_encoded = pd.get_dummies(df, columns=['Item', 'Mode of Payment'])
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6728 entries, 0 to 6727
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Date             6728 non-null   object 
 1   Item             6728 non-null   object 
 2   Price            6728 non-null   float64
 3   Quantity         6728 non-null   float64
 4   Mode of Payment  6728 non-null   object 
 5   ItemEncoded      6728 non-null   int8   
 6   PaymentEncoded   6728 non-null   int8   
dtypes: float64(2), int8(2), object(3)
memory usage: 276.1+ KB
None


In [13]:
import numpy as np

def generate_digital_twin(num_samples):
    noise = np.random.normal(0, 1, size=(num_samples, latent_dim))
    generated_data = generator.predict(noise)
    
    # Get original category labels
    item_categories = df['Item'].astype('category').cat.categories
    payment_categories = df['Mode of Payment'].astype('category').cat.categories
    
    # Create a separate array for decoded categories
    decoded_categories = np.empty((num_samples, 2), dtype=object)  # dtype=object to hold strings
    decoded_categories[:, 0] = item_categories[generated_data[:, 0].astype(int)]
    decoded_categories[:, 1] = payment_categories[generated_data[:, 1].astype(int)]
    
    # Inverse normalize numerical features
    decoded_numericals = np.empty((num_samples, 2))
    decoded_numericals[:, 0] = (generated_data[:, 2] * df['Price'].std()) + df['Price'].mean()
    decoded_numericals[:, 1] = (generated_data[:, 3] * df['Quantity'].std()) + df['Quantity'].mean()
    
    # Combine the categories and numericals into a single structured array for output, if necessary
    combined_data = np.hstack((decoded_categories, decoded_numericals))
    print(combined_data)
    return combined_data

In [14]:
num_samples = 100
digital_twin = generate_digital_twin(num_samples)

4/4 [==============================] - 0s 2ms/step
[['almond extract' 'Cash' 1.0 0.999856173992157]
 ['almond extract' 'Cash' 1.0 0.9994147419929504]
 ['almond extract' 'Cash' 0.9961585998535156 -0.1201583594083786]
 ['almond extract' 'Cash' 1.0 0.9977036714553833]
 ['almond extract' 'Cash' 0.9981203675270081 0.8526570796966553]
 ['almond extract' 'Cash' 0.9999473094940186 0.87110835313797]
 ['almond extract' 'Cash' 0.9999976754188538 0.988078236579895]
 ['almond extract' 'Cash' 1.0 0.9976010322570801]
 ['almond extract' 'Cash' 1.0 0.9989160895347595]
 ['almond extract' 'Cash' 1.0 0.9963951706886292]
 ['almond extract' 'Cash' 0.9997166991233826 0.04789479821920395]
 ['almond extract' 'Cash' 0.9999845027923584 0.06022031232714653]
 ['almond extract' 'Cash' 0.9997804164886475 -0.5955732464790344]
 ['almond extract' 'Cash' 0.9998040199279785 0.5130088329315186]
 ['almond extract' 'Cash' 0.9984297752380371 0.850334107875824]
 ['almond extract' 'Cash' 1.0 0.9948024749755859]
 ['almond extra

In [16]:
digital_twin[:, 2] = (digital_twin[:, 2] * df['Price'].std()) + df['Price'].mean()

In [17]:
digital_twin[:, 3] = (digital_twin[:, 3] * df['Quantity'].std()) + df['Quantity'].mean()

In [18]:
print(digital_twin)

[['almond extract' 'Cash' 0.9999999999999997 0.999856173992157]
 ['almond extract' 'Cash' 0.9999999999999997 0.9994147419929504]
 ['almond extract' 'Cash' 0.9961585998535153 -0.12015835940837856]
 ['almond extract' 'Cash' 0.9999999999999997 0.9977036714553833]
 ['almond extract' 'Cash' 0.9981203675270077 0.8526570796966553]
 ['almond extract' 'Cash' 0.9999473094940182 0.87110835313797]
 ['almond extract' 'Cash' 0.9999976754188534 0.988078236579895]
 ['almond extract' 'Cash' 0.9999999999999997 0.9976010322570801]
 ['almond extract' 'Cash' 0.9999999999999997 0.9989160895347595]
 ['almond extract' 'Cash' 0.9999999999999997 0.9963951706886292]
 ['almond extract' 'Cash' 0.9997166991233822 0.047894798219203984]
 ['almond extract' 'Cash' 0.9999845027923581 0.060220312327146565]
 ['almond extract' 'Cash' 0.9997804164886471 -0.5955732464790344]
 ['almond extract' 'Cash' 0.9998040199279782 0.5130088329315186]
 ['almond extract' 'Cash' 0.9984297752380368 0.850334107875824]
 ['almond extract' 'Cas

In [6]:
item_categories = df['Item'].astype('category').cat.categories
print(item_categories)


Index(['almond extract', 'apples', 'avocado', 'bacon', 'bananas',
       'barbecue sauce', 'beef', 'bell peppers', 'bouillon cubes', 'bread',
       ...
       'tortilla chips', 'tortillas', 'tuna', 'vanilla extract', 'vegetables',
       'vinegar', 'waffle mix', 'water', 'yogurt', 'zucchini'],
      dtype='object', length=124)


In [7]:
payment_categories = df['Mode of Payment'].astype('category').cat.categories
print(payment_categories)

Index(['Cash', 'Credit Card', 'Debit Card'], dtype='object')
